**Set environment**

In [1]:
suppressMessages(suppressWarnings(source("../run_config_project_sing.R")))
show_env()

You are working on        Singularity: singularity_proj_encode_fcc 
BASE DIRECTORY (FD_BASE): /net/seq/data2/projects/reddylab/Kuei 
REPO DIRECTORY (FD_REPO): /net/seq/data2/projects/reddylab/Kuei/repo 
WORK DIRECTORY (FD_WORK): /net/seq/data2/projects/reddylab/Kuei/work 
DATA DIRECTORY (FD_DATA): /net/seq/data2/projects/reddylab/Kuei/data 

You are working with      ENCODE FCC 
PATH OF PROJECT (FD_PRJ): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC 
PROJECT RESULTS (FD_RES): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/results 
PROJECT SCRIPTS (FD_EXE): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/scripts 
PROJECT DATA    (FD_DAT): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/data 
PROJECT NOTE    (FD_NBK): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/notebooks 
PROJECT DOCS    (FD_DOC): /net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/docs 
PROJECT LOG     (FD_LOG): /net/seq/data2/projects/reddylab/Kuei/re

## Prepare

**Set global variable**

In [2]:
vec = c(
    "fcc_astarr_macs_input_overlap",
    "fcc_astarr_macs_input_union"
)
names(vec) = vec

VEC_TXT_FOLDER = vec
for(txt in vec){cat(txt, "\n")}

fcc_astarr_macs_input_overlap 
fcc_astarr_macs_input_union 


**View files: FCC region coverage**

In [3]:
txt_folder = VEC_TXT_FOLDER[1]
txt_fdiry  = file.path(FD_RES, "analysis_fcc_correlation", txt_folder)
txt_fname  = "region.coverage.score.*.tsv"
txt_fglob  = file.path(txt_fdiry, txt_fname)

vec = Sys.glob(txt_fglob)
for (txt in vec) {cat(txt, "\n")}

/net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/analysis_fcc_correlation/fcc_astarr_macs_input_overlap/region.coverage.score.concat.tsv 


**View files: STARR/MPRA CREs**

In [4]:
txt_fdiry = file.path(FD_RES, "region_annotation", "*", "summary")
txt_fname = "region.annotation.fcc_starrmpra_junke.group.tsv"
txt_fglob = file.path(txt_fdiry, txt_fname)

vec = Sys.glob(txt_fglob)
for(txt in vec){cat(txt, "\n")}

/net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/region_annotation/fcc_astarr_macs_input_overlap/summary/region.annotation.fcc_starrmpra_junke.group.tsv 
/net/seq/data2/projects/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/region_annotation/fcc_astarr_macs_input_union/summary/region.annotation.fcc_starrmpra_junke.group.tsv 


## Import data

### Import FCC region coverage scores

In [5]:
### loop to import data
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### set file directory
    txt_fdiry  = file.path(FD_RES, "analysis_fcc_correlation", txt_folder)
    txt_fname = "region.coverage.score.concat.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)

    ### read table
    dat = read_tsv(txt_fpath, show_col_types = FALSE)
    return(dat)
})

### assign and show
lst_dat_region_score_fcc_import = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
head(dat, 3)

$fcc_astarr_macs_input_overlap
[1] 586004      8

$fcc_astarr_macs_input_union
[1] 889739      8



Chrom,ChromStart,ChromEnd,Region,Score,Score_Label,Assay_Group,Assay_Label
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>
chr1,10038,10405,chr1:10038-10405,-3.5573621,Scale(Log2FC),ASTARR_KS91,ASTARR
chr1,14282,14614,chr1:14282-14614,0.0361388,Scale(Log2FC),ASTARR_KS91,ASTARR
chr1,16025,16338,chr1:16025-16338,-2.1376311,Scale(Log2FC),ASTARR_KS91,ASTARR


**Check: Count table**

In [6]:
lst = lst_dat_region_score_fcc_import
lst = lapply(lst, function(dat){
    res = table(dat$Assay_Label, dnn=c("Assay"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Region")
dat = dat %>% tidyr::spread(Region, Freq)
fun_display_table(dat)

Assay,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
ASTARR,150040,246850
ATAC,150041,246852
CRISPRi-Growth,72743,80288
CRISPRi-HCRFF,925,1330
ENCODE-rE2G Benchmark,3035,3169
LMPRA,61478,68497
TMPRA,1148,1722
WSTARR,146594,241031


### Import STARR/MPRA/CRISPR CREs

In [7]:
txt_folder = VEC_TXT_FOLDER[1]
txt_fdiry = file.path(FD_RES, "region_annotation", txt_folder, "summary")
txt_fname = "region.annotation.fcc_starrmpracrispr.*"
#txt_fname = "region.annotation.fcc_starrmpracrispr.assayvote.filtered.tsv"
txt_fglob = file.path(txt_fdiry, txt_fname)

vec_txt_fpath = Sys.glob(txt_fglob)
vec_txt_fname = basename(vec_txt_fpath)
for (txt in vec_txt_fname){cat(txt, "\n")}

region.annotation.fcc_starrmpracrispr.assayvote.filtered.tsv 
region.annotation.fcc_starrmpracrispr.assayvote.tsv 
region.annotation.fcc_starrmpracrispr.concat.tsv 


In [8]:
### loop to import data
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### set file directory
    txt_fdiry = file.path(FD_RES, "region_annotation", txt_folder, "summary")
    txt_fname = "region.annotation.fcc_starrmpracrispr.assayvote.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)

    ### read table
    dat = read_tsv(txt_fpath, show_col_types = FALSE)
    return(dat)
})

### assign and show
lst_dat_region_annot_fcc_starrmpracrispr_cres_import = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 100454      6

$fcc_astarr_macs_input_union
[1] 135780      6



Chrom,ChromStart,ChromEnd,Region,Num_Assay,Assays
chr1,10038,10405,chr1:10038-10405,1,ASTARR
chr1,16025,16338,chr1:16025-16338,1,ASTARR
chr1,17288,17689,chr1:17288-17689,1,ASTARR


**Check: Count table**

In [9]:
lst = lst_dat_region_annot_fcc_starrmpracrispr_cres_import
lst = lapply(lst, function(dat){
    res = table(dat$Num_Assay, dnn=c("Group"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Region")
dat = dat %>% tidyr::spread(Region, Freq)
fun_display_table(dat)

Group,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
1,66501,95992
2,22994,28122
3,9740,10384
4,1177,1237
5,31,34
6,6,6
7,5,5


## Arrange table

In [10]:
lst = lst_dat_region_score_fcc_import
names(lst)

[1] "fcc_astarr_macs_input_overlap" "fcc_astarr_macs_input_union"

In [11]:
lst = lst_dat_region_annot_fcc_starrmpracrispr_cres_import
names(lst)

[1] "fcc_astarr_macs_input_overlap" "fcc_astarr_macs_input_union"

In [12]:
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### get all regions
    dat = lst_dat_region_score_fcc_import[[txt_folder]]
    vec = unique(dat$Region)
    dat_region_score = dat
    vec_txt_region = vec

    ### get assayvote CREs
    dat = lst_dat_region_annot_fcc_starrmpracrispr_cres_import[[txt_folder]]
    dat = dat %>% dplyr::select(Region, Num_Assay) %>% dplyr::distinct()
    dat_region_annot = dat

    ### arrange
    dat = tibble(Region = vec_txt_region) %>%
        dplyr::left_join(dat_region_annot, by = "Region") %>%
        dplyr::mutate(Num_Assay = tidyr::replace_na(Num_Assay, 0)) %>%
        dplyr::mutate(Group = ifelse(Num_Assay < 4, paste0("#Assay = ", Num_Assay), "#Assay >= 4"))

    ###
    dat = dat_region_score %>% dplyr::left_join(dat, by = "Region")
    return(dat)
})

### assign and show
lst_dat_region_merge_fcc = lst
#lst_dat_region_annot_fcc_starrmpracrispr_cres_arrange = lst
dat = lst[[1]]
fun_display_table(head(dat, 3))

Chrom,ChromStart,ChromEnd,Region,Score,Score_Label,Assay_Group,Assay_Label,Num_Assay,Group
chr1,10038,10405,chr1:10038-10405,-3.5573621,Scale(Log2FC),ASTARR_KS91,ASTARR,1,#Assay = 1
chr1,14282,14614,chr1:14282-14614,0.0361388,Scale(Log2FC),ASTARR_KS91,ASTARR,0,#Assay = 0
chr1,16025,16338,chr1:16025-16338,-2.1376311,Scale(Log2FC),ASTARR_KS91,ASTARR,1,#Assay = 1


In [13]:
lst = lst_dat_region_merge_fcc
lst = lapply(lst, function(dat){
    res = table(dat$Num_Assay, dnn=c("Group"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Region")
dat = dat %>% tidyr::spread(Region, Freq)
fun_display_table(dat)

Group,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
0,176618,363306
1,252792,347505
2,101937,120878
3,48243,51305
4,6122,6434
5,206,224
6,47,47
7,39,40


In [14]:
lst = lst_dat_region_merge_fcc
lst = lapply(lst, function(dat){
    dat = dat %>% dplyr::select(Region, Group) %>% dplyr::distinct()
    res = table(dat$Group, dnn=c("Group"))
    dat = as.data.frame(res)
    return(dat)
})

dat = bind_rows(lst, .id = "Region")
dat = dat %>% tidyr::spread(Region, Freq)
fun_display_table(dat)

Group,fcc_astarr_macs_input_overlap,fcc_astarr_macs_input_union
#Assay = 0,49587,111072
#Assay = 1,66501,95992
#Assay = 2,22994,28122
#Assay = 3,9740,10384
#Assay >= 4,1219,1282


## Calculate Spearman correlations across assays with different region sets

**Set assays**

In [15]:
lst = lst_dat_region_merge_fcc
dat = lst[[1]]
unique(dat$Assay_Label)

[1] "ASTARR"                "WSTARR"                "TMPRA"                
[4] "LMPRA"                 "CRISPRi-Growth"        "CRISPRi-HCRFF"        
[7] "ENCODE-rE2G Benchmark" "ATAC"

In [16]:
vec = c("ATAC", "ASTARR", "LMPRA", "TMPRA", "WSTARR", "CRISPRi-Growth", "CRISPRi-HCRFF", "ENCODE-rE2G Benchmark")
names(vec) = vec

vec_txt_assay = vec

**Get assay pairs**

In [17]:
vec = vec_txt_assay
dat = combn(vec, 2)
dat = data.frame(dat)

dat_assay_comb2 = dat
fun_display_table(dat)

X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,X25,X26,X27,X28
ATAC,ATAC,ATAC,ATAC,ATAC,ATAC,ATAC,ASTARR,ASTARR,ASTARR,ASTARR,ASTARR,ASTARR,LMPRA,LMPRA,LMPRA,LMPRA,LMPRA,TMPRA,TMPRA,TMPRA,TMPRA,WSTARR,WSTARR,WSTARR,CRISPRi-Growth,CRISPRi-Growth,CRISPRi-HCRFF
ASTARR,LMPRA,TMPRA,WSTARR,CRISPRi-Growth,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,LMPRA,TMPRA,WSTARR,CRISPRi-Growth,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,TMPRA,WSTARR,CRISPRi-Growth,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,WSTARR,CRISPRi-Growth,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,CRISPRi-Growth,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,CRISPRi-HCRFF,ENCODE-rE2G Benchmark,ENCODE-rE2G Benchmark


**Loop through each combination of assays**

In [18]:
lst = lst_dat_region_merge_fcc
lst = lapply(lst, function(dat_region_merge){
    
    ### Loop through each combination of assays
    lst = lapply(dat_assay_comb2, function(vec_txt_assay_pair){
        
        ### get assay pair
        txt_assay1 = vec_txt_assay_pair[1]
        txt_assay2 = vec_txt_assay_pair[2]
        
        ### filter by assay
        dat = dat_region_merge
        dat = dat %>% 
            dplyr::filter(Assay_Label %in% vec_txt_assay_pair) %>%
            dplyr::select(Region, Group, Score, Assay_Label) %>% 
            tidyr::spread(Assay_Label, Score) %>%
            na.omit
        
        ### split by region group
        lst = split(dat, dat$Group)
        lst = lapply(lst, function(dat){
            ### get scores foreach assay
            vec1 = dat[[txt_assay1]]
            vec2 = dat[[txt_assay2]]

            ### calculate correlation and summarize
            num  = cor(vec1, vec2, method = "spearman")
            res  = data.frame(
                "Assay1" = txt_assay1,
                "Assay2" = txt_assay2,
                "scor"   = num,
                "size"   = length(vec1)
            )
            return(res)
        })

        ### concat across region groups
        dat = bind_rows(lst, .id = "Group")
        return(dat)
    })

    ### concat across assay pairs
    dat = bind_rows(lst)
    return(dat)
})

### assign and show
lst_dat_assay_scor = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 140   5

$fcc_astarr_macs_input_union
[1] 140   5



Group,Assay1,Assay2,scor,size
#Assay = 0,ATAC,ASTARR,0.1093293,49586
#Assay = 1,ATAC,ASTARR,0.1604995,66501
#Assay = 2,ATAC,ASTARR,0.1171428,22994


**Add region counts for future labeling**

In [19]:
lst = lapply(VEC_TXT_FOLDER, function(txt_folder){
    ### get the correlation
    dat_score = lst_dat_assay_scor[[txt_folder]]

    ### get the total count of region
    dat = lst_dat_region_merge_fcc[[txt_folder]]
    dat = dat %>%
        dplyr::select(Region, Group) %>%
        dplyr::distinct() %>%
        dplyr::count(Group, name = "Total")
    dat_count = dat

    ### merge
    dat = dplyr::right_join(dat_count, dat_score, by = "Group")
    return(dat)
})

### assign and show
lst_dat_assay_scor_arrange = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$fcc_astarr_macs_input_overlap
[1] 140   6

$fcc_astarr_macs_input_union
[1] 140   6



Group,Total,Assay1,Assay2,scor,size
#Assay = 0,49587,ATAC,ASTARR,0.1093293,49586
#Assay = 0,49587,ATAC,LMPRA,0.0874381,13488
#Assay = 0,49587,ATAC,TMPRA,0.0964442,120


## Export results

In [20]:
for (txt_folder in VEC_TXT_FOLDER){
    ### get tables
    dat_result = lst_dat_assay_scor_arrange[[txt_folder]]
    
    ### set file directory
    txt_fdiry  = file.path(FD_RES, "analysis_fcc_comparison", txt_folder)
    txt_fname = "result.comparison.fcc_starrmpracrispr.assayvote.correlation.fcc.tsv"
    txt_fpath = file.path(txt_fdiry, txt_fname)
    
    ### write table
    dat = dat_result
    write_tsv(dat, txt_fpath)
}